In [22]:
import os, json, time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [23]:
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

In [24]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [26]:
import pandas as pd
import numpy as np
from pathlib import Path


# ================== 0) ورودی‌ها ==================
model_ans = pd.read_csv(r"F:\Thesis\project\1-BaselineTest\Models Responses\few-shot-cot-e2p\deepseek r1t2 chimera\eval_baseline_report\merged.csv")  # شامل: id, answer, (اختیاری: model, confidence, latency_ms, tokens..., explanation...)
gold      = pd.read_csv(r"F:\Thesis\project\1-BaselineTest\GOLD\Gold.csv")  # شامل: idx, gold

model_id_col  = "id"
model_ans_col = "answer"
gold_id_col   = "idx"
gold_ans_col  = "Gold"

# ================== 1) نرمال‌سازی پاسخ‌ها و Gold ==================
def norm_ans(x):
    if pd.isna(x): return None
    s = str(x).strip().strip('"').strip("'")
    try:
        v = int(float(s))
        return str(v) if v in {1,2,3,4} else None
    except:
        for d in ["1","2","3","4"]:
            if s.endswith(d): return d
        return None

def parse_gold_single_or_pair(x):
    """
    Gold ممکن است در یک ردیف (مثلاً idx=52) به صورت '4-2' باشد.
    این تابع همان را به مجموعه پاسخ‌های مجاز تبدیل می‌کند؛ سایر ردیف‌ها تک‌گزینه‌ای هستند.
    """
    if pd.isna(x): 
        return set()
    s = str(x).strip()
    if "-" in s:
        parts = [p.strip() for p in s.split("-")]
        return {norm_ans(p) for p in parts if norm_ans(p) is not None}
    v = norm_ans(s)
    return {v} if v is not None else set()

model = model_ans.copy()
gold2 = gold.copy()

model[model_ans_col] = model[model_ans_col].apply(norm_ans)
gold2["gold_set"] = gold2[gold_ans_col].apply(parse_gold_single_or_pair)
gold2["is_multi"] = gold2["gold_set"].apply(lambda s: len(s) > 1)

# ================== 2) ادغام id↔idx ==================
df = pd.merge(
    model,
    gold2[[gold_id_col, "gold_set", "is_multi"]],
    left_on=model_id_col,
    right_on=gold_id_col,
    how="inner"
)

# صحت (lenient: هر عضو gold_set قابل قبول است)
df["correct"] = df.apply(lambda r: (r[model_ans_col] in r["gold_set"]) if r[model_ans_col] is not None else False, axis=1).astype(int)

# ================== 3) تنظیمات ذخیره خروجی‌ها ==================
root = Path("eval_report")
root.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Arial'

# ================== 4) eval_accuracy (پوشه اول) ==================
acc_dir = root / "eval_accuracy"
acc_dir.mkdir(exist_ok=True)

acc = df["correct"].mean()*100 if len(df) else 0.0
(acc_dir / "accuracy_overall.txt").write_text(f"Accuracy: {acc:.2f}%\n", encoding="utf-8")

by_pred = df.groupby(model_ans_col)["correct"].agg(["mean","count"]).reset_index()
by_pred["accuracy_%"] = by_pred["mean"]*100
by_pred.drop(columns=["mean"]).to_csv(acc_dir / "accuracy_by_pred.csv", index=False, encoding="utf-8-sig")

conf = pd.crosstab(df[model_ans_col], df["gold_set"].apply(lambda s: sorted(list(s))[0] if len(s)>0 else None),
                   rownames=["pred"], colnames=["gold_primary"], dropna=False).fillna(0).astype(int)
conf.to_csv(acc_dir / "confusion.csv", encoding="utf-8-sig")

df.to_csv(acc_dir / "eval_detailed.csv", index=False, encoding="utf-8-sig")

# نمودارها
plt.figure(figsize=(4,4))
plt.bar(["Accuracy"], [acc], color="#4E79A7"); plt.ylim(0,100)
plt.title("Baseline Accuracy"); plt.ylabel("Accuracy (%)")
plt.text(0, acc+1, f"{acc:.1f}%", ha="center", va="bottom", fontweight="bold")
plt.tight_layout(); plt.savefig(acc_dir / "plot_accuracy_overall.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

tmp = by_pred.copy()
tmp[model_ans_col] = tmp[model_ans_col].astype(str)
tmp = tmp.sort_values(model_ans_col)
plt.figure(figsize=(6,4))
plt.bar(tmp[model_ans_col], tmp["accuracy_%"], color="#F28E2B"); plt.ylim(0,100)
plt.xlabel("Predicted option"); plt.ylabel("Accuracy (%)"); plt.title("Accuracy by Predicted Option")
for x,y in zip(tmp[model_ans_col], tmp["accuracy_%"]): plt.text(x, y+1, f"{y:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.savefig(acc_dir / "plot_accuracy_by_pred.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

plt.figure(figsize=(5,4))
sns.heatmap(conf, annot=True, fmt="d", cmap="Blues", cbar=False, linewidths=1, linecolor="white")
plt.title("Confusion Matrix (pred x gold)")
plt.tight_layout(); plt.savefig(acc_dir / "plot_confusion_heatmap.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

# ================== 5) calibration_confidence (پوشه دوم) ==================
cal_dir = root / "calibration_confidence"
cal_dir.mkdir(exist_ok=True)

if "confidence" in df.columns and df["confidence"].notna().any():
    calib = df.groupby("confidence")["correct"].agg(["mean","count"]).reset_index()
    calib.rename(columns={"mean":"accuracy"}, inplace=True); calib["accuracy_%"] = calib["accuracy"]*100
    calib.to_csv(cal_dir / "calibration_by_conf.csv", index=False, encoding="utf-8-sig")

    # reliability curve
    plt.figure(figsize=(5,4))
    plt.plot(calib["confidence"], calib["accuracy_%"], marker="o", color="#4E79A7", label="Observed")
    plt.plot([1,5],[20,100], linestyle="--", color="gray", label="Ideal (scaled)")  # فقط راهنما
    plt.xticks([1,2,3,4,5]); plt.ylim(0,100)
    plt.xlabel("Confidence"); plt.ylabel("Accuracy (%)"); plt.title("Reliability Curve"); plt.legend()
    plt.tight_layout(); plt.savefig(cal_dir / "plot_reliability_curve.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    # trade-off accuracy vs coverage
    rows=[]
    for t in [1,2,3,4,5]:
        sub = df[df["confidence"]>=t]
        coverage = len(sub)/len(df)*100 if len(df) else 0.0
        acc_t = sub["correct"].mean()*100 if len(sub) else np.nan
        rows.append({"threshold":t,"coverage_%":coverage,"accuracy_%":acc_t,"n":len(sub)})
    pd.DataFrame(rows).to_csv(cal_dir / "threshold_tradeoff.csv", index=False, encoding="utf-8-sig")

# ================== 6) tokens_usage (پوشه سوم) ==================
tok_dir = root / "tokens_usage"
tok_dir.mkdir(exist_ok=True)
tok_cols = [c for c in ["prompt_tokens","completion_tokens","total_tokens"] if c in df.columns]

if tok_cols:
    df[tok_cols].agg(["mean","median","min","max"]).round(2).to_csv(tok_dir / "tokens_summary.csv", encoding="utf-8-sig")
    df.groupby("correct")[tok_cols].mean().round(2).to_csv(tok_dir / "tokens_by_correct.csv", encoding="utf-8-sig")

    if "completion_tokens" in df.columns and df["completion_tokens"].notna().any():
        plt.figure(figsize=(6,4))
        plt.hist(df["completion_tokens"].dropna(), bins=20, color="#59A14F", edgecolor="black")
        plt.xlabel("Completion tokens"); plt.ylabel("Frequency"); plt.title("Completion Tokens Distribution")
        plt.tight_layout(); plt.savefig(tok_dir / "plot_tokens_hist.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    if "completion_tokens" in df.columns and "latency_ms" in df.columns and df["completion_tokens"].notna().any() and df["latency_ms"].notna().any():
        plt.figure(figsize=(6,4))
        colors = df["correct"].map({1:"#59A14F",0:"#E15759"})
        plt.scatter(df["completion_tokens"], df["latency_ms"], c=colors, alpha=0.6, edgecolors="black", linewidths=0.3)
        plt.xlabel("Completion tokens"); plt.ylabel("Latency (ms)"); plt.title("Completion Tokens vs Latency")
        plt.tight_layout(); plt.savefig(tok_dir / "plot_tokens_vs_latency.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

# ================== 7) explanation_quality (پوشه چهارم) ==================
exp_dir = root / "explanation_quality"
exp_dir.mkdir(exist_ok=True)

if "explain_len_words" in df.columns and df["explain_len_words"].notna().any():
    df.groupby("correct")["explain_len_words"].agg(["mean","median","count"]).round(2).reset_index().to_csv(exp_dir / "explain_len_vs_correct.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(5,4))
    sns.boxplot(data=df, x="correct", y="explain_len_words", palette=["#E15759","#59A14F"])
    plt.xticks([0,1], ["Incorrect","Correct"]); plt.xlabel(""); plt.ylabel("Explanation length (words)")
    plt.title("Explanation Length vs Correctness")
    plt.tight_layout(); plt.savefig(exp_dir / "plot_explain_len_box.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    if "confidence" in df.columns and df["confidence"].notna().any():
        plt.figure(figsize=(6,4))
        plt.scatter(df["explain_len_words"], df["confidence"], alpha=0.6, color="#AF7AC5", edgecolors="black", linewidths=0.3)
        plt.xlabel("Explanation length (words)"); plt.ylabel("Confidence"); plt.title("Explain Length vs Confidence")
        plt.tight_layout(); plt.savefig(exp_dir / "plot_explain_len_vs_confidence.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

# ================== 8) latency_profile (پوشه پنجم) ==================
lat_dir = root / "latency_profile"
lat_dir.mkdir(exist_ok=True)

if "latency_ms" in df.columns and df["latency_ms"].notna().any():
    lat_summary = df["latency_ms"].agg(["mean","median","min","max",lambda s: np.percentile(s.dropna(),90), lambda s: np.percentile(s.dropna(),95)]).round(1)
    lat_summary.index = ["mean","median","min","max","p90","p95"]
    lat_summary.to_csv(lat_dir / "latency_summary.csv", header=False, encoding="utf-8-sig")

    plt.figure(figsize=(6,4))
    plt.hist(df["latency_ms"].dropna(), bins=20, color="#4E79A7", edgecolor="black")
    plt.xlabel("Latency (ms)"); plt.ylabel("Frequency"); plt.title("Latency Distribution")
    plt.tight_layout(); plt.savefig(lat_dir / "plot_latency_dist.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    if "confidence" in df.columns and df["confidence"].notna().any():
        plt.figure(figsize=(6,4))
        plt.scatter(df["confidence"], df["latency_ms"], alpha=0.6, color="#E15759", edgecolors="black", linewidths=0.3)
        plt.xlabel("Confidence"); plt.ylabel("Latency (ms)"); plt.title("Confidence vs Latency")
        plt.tight_layout(); plt.savefig(lat_dir / "plot_conf_vs_latency.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

# ================== 9) error_inspection (پوشه ششم) ==================
err_dir = root / "error_inspection"
err_dir.mkdir(exist_ok=True)

# رایج‌ترین خطاها (pred→gold_primary)
err_pairs = conf.stack().reset_index(name="count").rename(columns={"pred":model_ans_col, "gold_primary":"gold"})
err_pairs = err_pairs[err_pairs[model_ans_col] != err_pairs["gold"]].sort_values("count", ascending=False)
err_pairs.head(20).to_csv(err_dir / "top_mistakes.csv", index=False, encoding="utf-8-sig")

# نمونه‌های مرزی (در صورت وجود confidence)
if "confidence" in df.columns and df["confidence"].notna().any():
    hard_false = df[(df["correct"]==0) & (df["confidence"]>=4)]
    lucky_true = df[(df["correct"]==1) & (df["confidence"]<=2)]
    hard_false.to_csv(err_dir / "review_hard_false_high_conf.csv", index=False, encoding="utf-8-sig")
    lucky_true.to_csv(err_dir / "review_lucky_true_low_conf.csv", index=False, encoding="utf-8-sig")

print(f"✅ All reports saved under: {root.resolve()}")

C:\Users\sazgar\AppData\Local\Temp\ipykernel_12008\2755134997.py:161: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="correct", y="explain_len_words", palette=["#E15759","#59A14F"])


✅ All reports saved under: F:\Thesis\project\1-BaselineTest\Models Responses\few-shot-cot-e2p\deepseek r1t2 chimera\eval_baseline_report\eval_report


In [27]:
nan_rows = df[df[model_ans_col].isna()].copy()

print(f"🔎 Nan preds: {len(nan_rows)} rows")
print(nan_rows[[model_id_col, "gold_set"]].head(10).to_string(index=False))

# لیست شناسه‌هایی که باید دوباره اجرا شوند
ids_to_rerun = nan_rows[model_id_col].tolist()


🔎 Nan preds: 2 rows
 id gold_set
 30      {1}
 89       {}


In [28]:
questions_path = r"F:\Thesis\project\403-vekalat\structured_questions.csv"
questions_df = pd.read_csv(questions_path)

subset_questions = questions_df[questions_df["question_number"].isin(ids_to_rerun)].copy()
print(f"🧪 Will re-run {len(subset_questions)} questions with empty preds")

🧪 Will re-run 2 questions with empty preds


In [29]:
def to_list(opts):
    if isinstance(opts, list): return opts
    if isinstance(opts, str):
        try:
            v = json.loads(opts)
            if isinstance(v, list): return v
        except Exception: pass
        for sep in ["|","؛",";","/","\\","،","\n"]:
            if sep in opts: return [x.strip() for x in opts.split(sep) if x.strip()]
        return [opts.strip()]
    return [str(opts)]

In [8]:
def render_numeric_options(opts):
    return "\n".join(f"{i+1}) {o}" for i, o in enumerate(opts))

In [30]:
def build_messages(question: str, options_text: str):
    return [
        {
            "role": "system",
            "content": (
                "# Iranian Legal Question Answering System\n"
                "You are a **professional legal reasoning assistant** specialized in **Iranian law**.\n"
                "---\n"
                "## Task:\n"
                "- Language: Persian (فارسی)\n"
                "- Objective: Analyze the question based on Iranian law and choose the **correct option number (1–4)**.\n"
                "- Provide a **short legal explanation (1–2 sentences in Persian)** describing:\n"
                "  1. Which legal article, principle, or rule applies.\n"
                "  2. Why that rule leads to the chosen answer.\n"
                "- Only explain **why the correct option is right** — do not analyze incorrect options.\n"
                "- After the explanation, output the final result in **valid JSON format**.\n"
                "---\n"
                "## Confidence Scale (5 Levels):\n"
                "- 1: UNCERTAIN — Cannot distinguish between two or more options.\n"
                "- 2: WEAK — Significant doubt or debatable interpretation.\n"
                "- 3: MODERATE — Fairly confident; based on a standard or commonly accepted interpretation.\n"
                "- 4: STRONG — Confident; clear legal article, rule, or precedent supports the choice.\n"
                "- 5: VERY STRONG — Certain; unambiguous and directly supported by law.\n"
                "---\n"
                "# ------------------ FEW-SHOT EXAMPLES ------------------\n"

                "\n# Example 1 (confidence=5)\n"
                "User: سؤال: کدام‌يک از مراجع ذکر شده در قانون اساسی، مرجع تخصصی قضايی محسوب می‌شود؟\n"
                "گزینه‌ها:\n"
                "1) دادستانی کل کشور\n"
                "2) دیوان محاسبات کشور\n"
                "3) دیوان عدالت اداری\n"
                "4) دیوان عالی کشور\n"
                "Assistant: {\"explanation\":\"براساس اصل 173 قانون اساسی، دیوان عدالت اداری مرجع تخصصی قضایی برای رسیدگی به شکایات مردم علیه مأمورین و آیین‌نامه‌های دولتی است.\",\"answer\":\"3\",\"confidence\":5}\n"

                "\n# Example 2 (confidence=4)\n"
                "User: سؤال: آیا وجود قرابت نسبی یا سببی بین وکلای طرفین دعوا در یک پرونده جریانی، موجب ممنوعیت از انجام وکالت در پرونده است؟\n"
                "گزینه‌ها:\n"
                "1) در صورت تسخیری بودن وکالت یکی از وکلا، قرابت تا درجه سوم از هر طبقه موجب ممنوعیت وکیل تسخیری از انجام وکالت در پرونده است.\n"
                "2) قرابت تا درجه سوم از هر طبقه بین وکلای تعیینى طرفین دعوا، موجب ممنوعیت از انجام وکالت در پرونده است.\n"
                "3) مطلقاً از موجبات ممنوعیت از انجام وکالت در پرونده است.\n"
                "4) از موجبات ممنوعیت از انجام وکالت در پرونده نیست.\n"
                "Assistant: {\"explanation\":\"مطابق ماده 349 قانون آیین دادرسی کیفری، قرابت تا درجه سوم فقط در صورت تسخیری بودن وکالت موجب ممنوعیت است.\",\"answer\":\"1\",\"confidence\":4}\n"

                "\n# Example 3 (confidence=3)\n"
                "User: سؤال: «الف» مبادرت به جعل چک تضمین‌شده می‌نماید و آن را به «ب» می‌دهد. «ب» پس از ارائه چک به بانک، متوجه جعلی بودن می‌شود. «الف» چه جرمی را مرتکب شده است؟\n"
                "گزینه‌ها:\n"
                "1) جعل و استفاده از سند مجعول و صدور چک بلامحل\n"
                "2) جعل و معاونت در استفاده از سند مجعول\n"
                "3) جعل و استفاده از سند مجعول\n"
                "4) جعل و صدور چک بلامحل\n"
                "Assistant: {\"explanation\":\"چون صدور چک بلامحل فقط در صورت صادرکننده واقعی صادقه است و استفاده نیز توسط جاعل انجام شده، جرم جعل و استفاده از سند مجعول تحقق دارد.\",\"answer\":\"3\",\"confidence\":3}\n"

                "\n# Example 4 (confidence=2)\n"
                "User: سؤال: اگر شخصی مالی را که در ید امانت است بفروشد ولی آن را تحویل ندهد، آیا مرتکب خیانت در امانت شده است؟\n"
                "گزینه‌ها:\n"
                "1) بله، صرف اقدام به فروش کافی است.\n"
                "2) خیر، تا زمانی که مال تسلیم نشود، جرم تحقق نمی‌یابد.\n"
                "3) بستگی به قصد مرتکب دارد.\n"
                "4) در هر حال خیانت در امانت محسوب می‌شود.\n"
                "Assistant: {\"explanation\":\"صرف فروش مال امانی بدون تصرف مادی هنوز موجب استیلای غیرقانونی نیست؛ تحقق جرم نیازمند تسلیم یا تصرف است.\",\"answer\":\"2\",\"confidence\":2}\n"

                "\n# Example 5 (confidence=1)\n"
                "User: سؤال: اگر دو شاهد در شهادت خود دچار تردید شوند ولی هر دو اصل وقوع جرم را تأیید کنند، آیا شهادت آنان معتبر است؟\n"
                "گزینه‌ها:\n"
                "1) معتبر است، چون اصل وقوع جرم را تأیید کرده‌اند.\n"
                "2) معتبر نیست، چون یقین و جزم در شهادت شرط است.\n"
                "3) فقط در امور مدنی معتبر است.\n"
                "4) بستگی به نوع جرم دارد.\n"
                "Assistant: {\"explanation\":\"مطابق ماده 1315 قانون مدنی، شهادت باید با جزم باشد؛ تردید موجب سقوط اعتبار است، هرچند در عمل اختلاف نظر وجود دارد.\",\"answer\":\"2\",\"confidence\":1}\n"
                "---\n"
                "## OUTPUT FORMAT:\n"
                "Return **only one JSON object**, formatted exactly like this:\n"
                '{"explanation":"توضیح کوتاه حقوقی شامل ماده یا قاعده و استدلال","answer":"X","confidence":Y}\n'
                "- explanation → short legal reasoning (in Persian, 1–2 sentences)\n"
                "- answer → one of \"1\",\"2\",\"3\",\"4\"\n"
                "- confidence → integer 1–5 according to the defined scale\n"
                "---\n"
                "⚠️ IMPORTANT:\n"
                "- Output **must start with `{` and end with `}`.\n"
                "- **Do not** include any text, markdown, or comments before or after the JSON."
            )
        },
        {
            "role": "user",
            "content": f"""Question:
{question}

Options:
{options_text}

Expected output format:
{{"explanation":"توضیح کوتاه حقوقی شامل ماده یا قاعده و استدلال","answer":"شماره_گزینه","confidence":میزان_اطمینان}}"""
        }
    ]


In [31]:
def extract_json(content: str, verbose: bool = False):
    """
    Extract the last valid JSON object from an LLM response.
    Supports schemas with keys: explanation, answer, confidence.
    """
    import json, re

    if not content or not str(content).strip():
        if verbose:
            print("⚠️ Content is empty or None")
        return None

    content = str(content).strip()

    # Remove markdown code fences ``````
    if content.startswith("```"):
        lines = content.split("\n")
        if lines and lines.strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        content = "\n".join(lines).strip()

    # Attempt 1: Direct parse (whole content is JSON)
    try:
        parsed = json.loads(content)
        if verbose:
            print(f"✓ Direct parse successful: {parsed}")
        return parsed
    except json.JSONDecodeError as e:
        if verbose:
            print(f"⚠️ Direct parse failed: {e}")

    # Attempt 2: Regex extraction of ALL JSON objects, return the LAST valid
    # This pattern tolerates nested braces one level; sufficient for flat objects.
    pattern = r'\{(?:[^{}]|(?:\{[^{}]*\}))*\}'
    candidates = [m.group(0).strip() for m in re.finditer(pattern, content, re.DOTALL)]

    last_valid = None
    for cand in candidates:
        try:
            obj = json.loads(cand)
            last_valid = obj  # keep overwriting to end up with the last one
        except json.JSONDecodeError:
            if verbose:
                print(f"⚠️ Invalid JSON candidate: {cand[:120]}...")
            continue

    if last_valid is not None:
        if verbose:
            print(f"✓ Regex extraction successful (last JSON): {last_valid}")
        return last_valid

    # Attempt 3: Manual fallback – extract fields individually
    # explanation (optional)
    expl_match = re.search(r'"explanation"\s*:\s*"(?P<exp>(?:\\.|[^"\$$)*)"', content)
    explanation = expl_match.group("exp").strip() if expl_match else ""

    # answer (required)
    ans_match = re.search(r'"answer"\s*:\s*"(?P<ans>\d+)"', content)
    # confidence (required)
    conf_match = re.search(r'"confidence"\s*:\s*(?P<conf>-?\d+(?:\.\d+)?)', content)

    if ans_match and conf_match:
        conf_raw = conf_match.group("conf")
        try:
            confidence = int(float(conf_raw))
        except Exception:
            confidence = None

        fallback = {
            "explanation": explanation,
            "answer": ans_match.group("ans"),
            "confidence": confidence
        }
        if verbose:
            print(f"✓ Manual extraction successful: {fallback}")
        return fallback

    if verbose:
        preview = content if len(content) < 200 else content[:200] + "..."
        print(f"⚠️ No valid JSON found in content:\n{preview}")
    return None


In [32]:
VALID_CONFIDENCE = {1, 2, 3, 4, 5}

def call_model(
    client,
    model: str, 
    messages, 
    temperature: float = 0.1, 
    max_tokens: int = 1024,
    valid_answers: set = None,
    verbose: bool = False,
    timeout: float = 60.0,
    stop: list | None = None,
):
    """
    Stable call: extracts explanation, answer, confidence, and token usage.
    Returns:
        dict: {
            "answer": str | None,
            "confidence": int | None,
            "explanation": str,
            "raw": str,
            "latency_ms": int | None,
            "prompt_tokens": int | None,
            "completion_tokens": int | None,
            "total_tokens": int | None,
            "explain_len_chars": int | None,
            "explain_len_words": int | None,
            "error": str | None
        }
    """
    if valid_answers is None:
        valid_answers = {"1", "2", "3", "4"}

    t0 = time.time()
    content = None

    try:
        # --- Model Call ---
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            timeout=timeout,
            stop=stop,
        )

        latency = int((time.time() - t0) * 1000)

        # --- Extract content safely ---
        choice0 = resp.choices[0]
        content = (
            getattr(choice0, "message", None).content if getattr(choice0, "message", None) else None
        ) or getattr(choice0, "text", None)

        if not content or not str(content).strip():
            raise ValueError("Model returned no text content")

        content = str(content).strip()

        # --- Usage tokens (may be None if provider doesn't return) ---
        usage = getattr(resp, "usage", None)
        prompt_tokens     = getattr(usage, "prompt_tokens", None) if usage else None
        completion_tokens = getattr(usage, "completion_tokens", None) if usage else None
        total_tokens      = getattr(usage, "total_tokens", None) if usage else None

        if verbose:
            finish_reason = getattr(choice0, "finish_reason", None)
            print(f"\n🔍 Model: {model} | Latency: {latency} ms | finish_reason: {finish_reason} | usage: {getattr(usage, '__dict__', usage)}")
            print(f"🔍 Raw output preview:\n{content[:200]}...\n")

        # --- Extract JSON structure (supports explanation/answer/confidence) ---
        data = extract_json(content, verbose=verbose)
        if not data:
            raise ValueError("Failed to extract valid JSON structure from model output")

        # --- Parse fields ---
        explanation = str(data.get("explanation", "") or "").strip()
        explain_len_chars = len(explanation) if explanation else 0
        explain_len_words = len(explanation.split()) if explanation else 0

        answer = str(data.get("answer", "")).strip().strip('"').strip("'")
        if answer not in valid_answers:
            raise ValueError(f"Invalid answer '{answer}', expected one of {valid_answers}")

        conf_raw = data.get("confidence", None)
        if conf_raw is None:
            raise ValueError("Missing 'confidence' field in model output")

        try:
            confidence = int(conf_raw)
        except Exception:
            try:
                confidence = int(float(str(conf_raw).strip()))
            except Exception:
                raise ValueError(f"Invalid confidence value: {conf_raw}")

        if confidence not in VALID_CONFIDENCE:
            original = confidence
            confidence = min(VALID_CONFIDENCE, key=lambda x: abs(x - original))
            if verbose:
                print(f"⚠️ Adjusted confidence from {original} → {confidence}")

        if verbose:
            print(f"✅ Parsed → answer={answer}, confidence={confidence}, explain_len(chars)={explain_len_chars}, tokens: p={prompt_tokens}, c={completion_tokens}, t={total_tokens}")

        return {
            "answer": answer,
            "confidence": confidence,
            "explanation": explanation,
            "raw": content,
            "latency_ms": latency,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
            "explain_len_chars": explain_len_chars,
            "explain_len_words": explain_len_words,
            "error": None,
        }

    except Exception as e:
        latency = int((time.time() - t0) * 1000)
        if verbose:
            print(f"❌ Error: {str(e)}")
            if content:
                print(f"❌ Raw content preview:\n{str(content)[:300]}...\n")

        return {
            "answer": None,
            "confidence": None,
            "explanation": "",
            "raw": content or "",
            "latency_ms": latency,
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "explain_len_chars": None,
            "explain_len_words": None,
            "error": str(e),
        }


In [33]:
def run_few_shot_e2p(
    client,
    model: str,
    df,
    limit=None,
    verbose: bool = False
):
    """
    Run few-shot evaluation on questions (single run per question).

    Returns:
        pd.DataFrame with columns:
            - id, model, answer, confidence, explanation, latency_ms,
              prompt_tokens, completion_tokens, total_tokens,
              explain_len_chars, explain_len_words,
              raw, error
    """
    rows = []
    errors = []

    iterator = tqdm(df.iterrows(), total=len(df), desc=f"Few-Shot({model})", disable=verbose)

    processed = 0
    for idx, row in iterator:
        if limit is not None and processed >= int(limit):
            break

        qid = row.get("question_number", idx)

        try:
            q = row["question"]
            opts_list = to_list(row["options"])
            options_text = render_numeric_options(opts_list)
            messages = build_messages(q, options_text)

            if verbose:
                print(f"\n📝 Q{qid}: {str(q)[:60]}...")

            # CoT به فضای بیشتری نیاز دارد
            result = call_model(
                client=client,
                model=model,
                messages=messages,
                temperature=0.1,
                max_tokens=1024,          # فضای کافی برای explain + JSON
                timeout=60.0,             # پایداری بهتر
                verbose=verbose
            )

            rec = {
                "id": qid,
                "model": model,
                "answer": result.get("answer"),
                "confidence": result.get("confidence"),
                "explanation": result.get("explanation", ""),
                "latency_ms": result.get("latency_ms"),
                "prompt_tokens": result.get("prompt_tokens"),
                "completion_tokens": result.get("completion_tokens"),
                "total_tokens": result.get("total_tokens"),
                "explain_len_chars": result.get("explain_len_chars"),
                "explain_len_words": result.get("explain_len_words"),
                "raw": result.get("raw", ""),
                "error": result.get("error"),
            }
            rows.append(rec)
            processed += 1

        except Exception as e:
            if verbose:
                print(f"❌ Q{qid}: {str(e)}")
            errors.append({"id": qid, "error": str(e)})

            rows.append({
                "id": qid,
                "model": model,
                "answer": None,
                "confidence": None,
                "explanation": "",
                "latency_ms": None,
                "prompt_tokens": None,
                "completion_tokens": None,
                "total_tokens": None,
                "explain_len_chars": None,
                "explain_len_words": None,
                "raw": "",
                "error": str(e)
            })
            processed += 1
            continue

    if errors and verbose:
        print(f"\n⚠️  {len(errors)} questions had errors")

    df_out = pd.DataFrame(rows)

    desired_cols = [
        "id","model","answer","confidence","explanation","latency_ms",
        "prompt_tokens","completion_tokens","total_tokens",
        "explain_len_chars","explain_len_words",
        "raw","error"
    ]
    df_out = df_out[[c for c in desired_cols if c in df_out.columns]]

    return df_out


In [34]:
# ================== شروع اجرا ==================
print("="*70)
print("🚀 Few-Shot Evaluation")
print("="*70)
print(f"📂 Dataset size: {len(subset_questions)} questions")
print(f"📊 Model: TNG: DeepSeek R1T2 Chimera")
print("="*70 + "\n")

start_time = time.time()

try:
    # اجرا
    results = run_few_shot_e2p(
        client=client,
        model="tngtech/deepseek-r1t2-chimera:free",
        df=subset_questions,
        limit=None,   # None = همه
        verbose=True # توصیه: False برای سرعت؛ چند تست اول را True کن
    )

    elapsed = time.time() - start_time

    # ================== خلاصه نتایج ==================
    print("\n" + "="*70)
    print("✅ Execution Completed!")
    print("="*70)
    print(f"📊 Processed: {len(results)} questions")
    print(f"⏱️  Total time: {elapsed/60:.1f} minutes ({elapsed:.1f} seconds)")
    print(f"⚡ Avg time per question: {elapsed/len(results):.2f} seconds")

    # ================== آمار کلیدی ==================
    print("\n" + "="*70)
    print("📈 Summary Statistics")
    print("="*70)
    
    # Confidence آمار
    if "confidence" in results.columns:
        valid_conf = results["confidence"].dropna()
        if len(valid_conf) > 0:
            avg_conf = valid_conf.mean()
            print(f"   Average confidence: {avg_conf:.1f}")
            print(f"   Median confidence: {valid_conf.median():.1f}")
            print(f"   Min confidence: {valid_conf.min():.1f}")
            print(f"   Max confidence: {valid_conf.max():.1f}")

            total = len(results)
            very_low = (valid_conf <= 2).sum()
            low      = ((valid_conf > 2) & (valid_conf <= 3)).sum()
            medium   = ((valid_conf > 3) & (valid_conf <= 4)).sum()
            high     = (valid_conf > 4).sum()

            print(f"\n   Very Low (≤2): {very_low} ({very_low/total*100:.1f}%)")
            print(f"   Low (2-3): {low} ({low/total*100:.1f}%)")
            print(f"   Medium (3-4): {medium} ({medium/total*100:.1f}%)")
            print(f"   High (>4): {high} ({high/total*100:.1f}%)")

    # توزیع پاسخ‌ها
    if "answer" in results.columns:
        print(f"\n   Answer distribution:")
        answer_dist = results["answer"].value_counts().sort_index()
        for ans, count in answer_dist.items():
            print(f"      {ans}: {count} ({count/len(results)*100:.1f}%)")

    # خطاها
    if "error" in results.columns:
        errors_count = (results["error"].notna()).sum()
        if errors_count > 0:
            print(f"\n   ⚠️  Questions with errors: {errors_count} ({errors_count/len(results)*100:.1f}%)")
        else:
            print(f"\n   ✅ No errors!")

    # Latency آمار
    if "latency_ms" in results.columns:
        valid_latency = results["latency_ms"].dropna()
        if len(valid_latency) > 0:
            print(f"\n   Average latency: {valid_latency.mean():.0f}ms")
            print(f"   Median latency:  {valid_latency.median():.0f}ms")
            print(f"   Max latency:     {valid_latency.max():.0f}ms")

    # ================== آمار مبتنی بر توکن‌ها ==================
    print("\n" + "="*70)
    print("🔎 Token-based Analysis")
    print("="*70)

    have_tokens = {"prompt_tokens","completion_tokens","total_tokens"}.issubset(results.columns)
    if have_tokens:
        # خلاصه
        for col in ["prompt_tokens","completion_tokens","total_tokens"]:
            s = results[col].dropna()
            if len(s) > 0:
                print(f"   {col}: mean={s.mean():.1f}, median={s.median():.1f}, min={s.min()}, max={s.max()}")

        # همبستگی با confidence
        if "confidence" in results.columns:
            subset = results.dropna(subset=["confidence","completion_tokens"])
            if len(subset) > 1:
                corr_p = subset[["completion_tokens","confidence"]].corr(method="pearson").iloc[0,1]
                corr_s = subset[["completion_tokens","confidence"]].corr(method="spearman").iloc[0,1]
                print(f"\n   Correlation (Pearson) completion_tokens ~ confidence: {corr_p:.3f}")
                print(f"   Correlation (Spearman) completion_tokens ~ confidence: {corr_s:.3f}")

        # همبستگی latency با توکن‌ها
        if "latency_ms" in results.columns:
            subset = results.dropna(subset=["latency_ms","completion_tokens"])
            if len(subset) > 1:
                corr_lp = subset[["completion_tokens","latency_ms"]].corr(method="pearson").iloc[0,1]
                print(f"   Correlation (Pearson) completion_tokens ~ latency_ms: {corr_lp:.3f}")

        # توکن‌ها به تفکیک پاسخ
        if "answer" in results.columns:
            by_ans = results.dropna(subset=["answer","completion_tokens"]).copy()
            if len(by_ans) > 0:
                print("\n   Completion tokens by answer (mean):")
                means = by_ans.groupby(by_ans["answer"].astype(str))["completion_tokens"].mean()
                for ans, m in means.sort_index().items():
                    print(f"      {ans}: {m:.1f}")

        # طول explain و اعتماد
        if {"explain_len_words","confidence"}.issubset(results.columns):
            sub = results.dropna(subset=["explain_len_words","confidence"])
            if len(sub) > 1:
                corr_ep = sub[["explain_len_words","confidence"]].corr(method="pearson").iloc[0,1]
                corr_es = sub[["explain_len_words","confidence"]].corr(method="spearman").iloc[0,1]
                print(f"\n   Correlation explain_len_words ~ confidence: Pearson={corr_ep:.3f}, Spearman={corr_es:.3f}")
    else:
        print("   ⚠️ Token usage columns not found. Ensure call_model returns prompt/completion/total tokens.")

    # ================== ذخیره نتایج ==================
    print("\n" + "="*70)
    print("💾 Saving Results")
    print("="*70)

    # CSV
    csv_filename = "results_few_shot_cot_e2p(subset).csv"
    results.to_csv(csv_filename, index=False, encoding="utf-8-sig")
    print(f"   ✅ CSV saved: {csv_filename}")

    # Excel
    excel_filename = "results_few_shot.xlsx"
    with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
        # Summary: ستون‌های کلیدی اگر موجود باشند
        summary_cols = [c for c in [
            "id","answer","confidence","explanation","latency_ms","error",
            "completion_tokens","prompt_tokens","total_tokens","explain_len_words"
        ] if c in results.columns]
        results[summary_cols].to_excel(writer, sheet_name="Summary", index=False)
        results.to_excel(writer, sheet_name="Details", index=False)
    print(f"   ✅ Excel saved: {excel_filename}")

    # ================== نمایش نمونه ==================
    print("\n" + "="*70)
    print("📋 Sample Results (First 10)")
    print("="*70)
    display_cols = [c for c in [
        "id","answer","confidence","explanation","latency_ms","error",
        "completion_tokens","prompt_tokens","total_tokens","explain_len_words"
    ] if c in results.columns]
    print(results[display_cols].head(10).to_string(index=False))

    # ================== سؤالات با خطا ==================
    if "error" in results.columns:
        error_rows = results[results["error"].notna()]
        if len(error_rows) > 0:
            print("\n" + "="*70)
            print("⚠️  Questions with Errors")
            print("="*70)
            print(error_rows[["id","error"]].head(10).to_string(index=False))

    print("\n" + "="*70)
    print("🎉 All Done!")
    print("="*70)

except KeyboardInterrupt:
    print("\n⚠️  Execution interrupted by user")
    if "results" in locals() and len(results) > 0:
        results.to_csv("results_partial.csv", index=False, encoding="utf-8-sig")
        print(f"   💾 Partial results saved: {len(results)} questions")

except Exception as e:
    print("\n" + "="*70)
    print("❌ ERROR During Execution")
    print("="*70)
    print(f"   Error type: {type(e).__name__}")
    print(f"   Error: {str(e)}")
    import traceback
    traceback.print_exc()

finally:
    total_elapsed = time.time() - start_time
    print(f"\n⏱️  Total execution time: {total_elapsed/60:.2f} minutes")
        

🚀 Few-Shot Evaluation
📂 Dataset size: 2 questions
📊 Model: TNG: DeepSeek R1T2 Chimera


📝 Q30: در خصوص قابلیت فرجام احکام صادره از دادگاه تجدید نظر راجع به...

🔍 Model: tngtech/deepseek-r1t2-chimera:free | Latency: 35402 ms | finish_reason: stop | usage: {'completion_tokens': 719, 'prompt_tokens': 1583, 'total_tokens': 2302, 'completion_tokens_details': None, 'prompt_tokens_details': PromptTokensDetails(audio_tokens=None, cached_tokens=2)}
🔍 Raw output preview:
{"explanation":"مطابق ماده 367 قانون آیین دادرسی مدنی، احکام راجع به اصل نکاح، فسخ آن، طلاق و نسب از جمله احکامی هستند که قابلیت فرجام خواهی دارند. این شامل متفرعات دعوی مرتبط با این موضوعات نیز می‌شو...

✓ Direct parse successful: {'explanation': 'مطابق ماده 367 قانون آیین دادرسی مدنی، احکام راجع به اصل نکاح، فسخ آن، طلاق و نسب از جمله احکامی هستند که قابلیت فرجام خواهی دارند. این شامل متفرعات دعوی مرتبط با این موضوعات نیز می\u200cشود.', 'answer': '2', 'confidence': 3}
✅ Parsed → answer=2, confidence=3, explain_len(chars)=186, 

In [36]:
import pandas as pd

main = pd.read_csv(r"F:\Thesis\project\1-BaselineTest\Models Responses\few-shot-cot-e2p\deepseek r1t2 chimera\eval_baseline_report\merged.csv")
sub  = pd.read_csv(r"F:\Thesis\project\1-BaselineTest\Models Responses\few-shot-cot-e2p\deepseek r1t2 chimera\eval_baseline_report\results_few_shot_cot_e2p(subset).csv")

# هم‌راستا کردن نوع id
main["id"] = main["id"].astype(str)
sub["id"]  = sub["id"].astype(str)

# اطمینان از یکتا بودن id در subset
if not sub["id"].is_unique:
    raise ValueError("id در فایل subset یکتا نیست؛ ابتدا آن را یکتا کن.")

# اگر subset ستون‌های جدیدی دارد که در main نیست، آن‌ها را هم به main اضافه می‌کنیم
for c in sub.columns:
    if c not in main.columns:
        main[c] = pd.NA

# جایگزینی همه ستون‌ها برای idهای مشترک
main_idx = main.set_index("id")
sub_idx  = sub.set_index("id")

# update فقط روی اندیس مشترک اعمال می‌شود و مقدارهای subset را روی main می‌نویسد
main_idx.update(sub_idx)

merged = main_idx.reset_index()

merged.to_csv(r"F:\Thesis\project\1-BaselineTest\Models Responses\few-shot-cot-e2p\deepseek r1t2 chimera\eval_baseline_report\results_few_shot_cot_e2p.csv",
              index=False, encoding="utf-8-sig")
print("✅ Merged and updated all matching rows by id.")


✅ Merged and updated all matching rows by id.
